In [ ]:
# ------------------------------------------------------------------------------------------------------------------------
# FANBEATS: Frequency and Attention-augmented Neural Basis Expansion Analysis for interpretable Time Series forecasting
#
# Copyright (c) 2026 Danish Abbas
#
# This file is part of the FANBEATS project.
# Licensed under the Creative Commons Attribution-NonCommercial
# 4.0 International License (CC BY-NC 4.0).
#
# See the LICENSE file in the root directory for full details.
# ------------------------------------------------------------------------------------------------------------------------

`Note: Before running this notebook and executing the code, please read the "README.md".`

# Preprocessing (Imputing missing values) the Solar Wind Speed data (from OMNI NASA)

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np 
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")

In [ ]:

datasets = Path(os.path.abspath("./"))

output = Path(os.path.abspath("./"))

In [ ]:
 #reading the original NASA solar wind speed dataset (please before running this and onward cells, make sure that you have the original dataset, for more info, please read README.md)
df = pd.read_csv("./org_solar_wind_speed_data.csv", header = None)
df.head()

In [ ]:

df = df.iloc[:, 0].str.split(expand=True)

df.columns = ["year", "day", "hour", "speed"]

df["year"] = df["year"].astype(int)
df["day"] = df["day"].astype(int)
df["hour"] = df["hour"].astype(int)
df["speed"] = pd.to_numeric(df["speed"], errors="coerce")

In [ ]:
df

In [ ]:
# 1- making the most important column of the dataset, which is the date_time column, by combining the year, day and hour columns
df["date_time"] = pd.to_datetime(df["year"].astype(str) + "-" + df["day"].astype(str) + " " + df["hour"].astype(str) + ":00:00", format="%Y-%j %H:%M:%S")
df

In [ ]:
#since now there is proper date_time, so dropping "year, day, and hour" columns
df.drop(["year", "day", "hour"], axis=1, inplace=True)
df

In [ ]:
df["id"] = 1 
df["frequency"] = "1h"

df

> Based on the Standard practice, keeping the train, val, and test dataset split
---

| Phase        | Period            | Samples | Sampling Freq |
|--------------|-------------------|---------|----------------|
| **Training** | 2011–2015         | 43,824  | Hourly         |
| **Validation** | 2016             | 8,784   | Hourly         |
| **Test**     | 2017              | 8,760   | Hourly         |

---

In [ ]:
 #making the phase placeholder column, which will have "train" label for values from 2011-01-01 00:00:00 to 2015-12-31 23:00:00, "val" label for values from 2016-01-01 00:00:00 to 2016-12-31 23:00:00, and "test" label for values from 2017-01-01 00:00:00 to 2017-12-31 23:00:00
df["phase"] = np.where((df["date_time"] >= "2011-01-01 00:00:00") & (df["date_time"] <= "2015-12-31 23:00:00"), "train", np.where((df["date_time"] >= "2016-01-01 00:00:00") & (df["date_time"] <= "2016-12-31 23:00:00"), "val", "test"))

df

In [ ]:

df["phase"].value_counts() 

In [ ]:
df["speed"] = df["speed"].replace(9999.0, np.nan)
df

In [ ]:
## Now most important thing is to split the dataset based on the specified date ranges and for imputation and saving them properly, based on phase
train_df = df[df["phase"] == "train"]
val_df = df[df["phase"] == "val"]   
test_df = df[df["phase"] == "test"]

In [ ]:
# now confirming the accurate splits and their sizes, with proper date_time ranges
print("Train dataset:")
print(f"Date range: {train_df['date_time'].min()} to {train_df['date_time'].max()}")
print(f"Number of samples: {len(train_df)}")
print("\nValidation dataset:")
print(f"Date range: {val_df['date_time'].min()} to {val_df['date_time'].max()}")
print(f"Number of samples: {len(val_df)}")
print("\nTest dataset:")
print(f"Date range: {test_df['date_time'].min()} to {test_df['date_time'].max()}")
print(f"Number of samples: {len(test_df)}")


In [ ]:

def impute_values(
    df: pd.DataFrame,
    target_col: str = "speed",
    verbose: bool = True,
) -> pd.DataFrame:

    df = df.copy()

    if target_col not in df.columns:
        raise KeyError(f"Column '{target_col}' not found in dataframe.")

    if verbose:
        print(f"[Before] Missing values in '{target_col}': {df[target_col].isna().sum()}")

    # 1. Main imputation
    df[target_col] = df[target_col].interpolate(
        method="linear",
        limit_direction="both"
    )

    # 2. Fallback cleanup
    df[target_col] = df[target_col].ffill().bfill()

    # 3. Final check
    remaining_nans = df[target_col].isna().sum()
    if remaining_nans > 0:
        raise ValueError(f"Still found {remaining_nans} NaNs in '{target_col}' after imputation.")

    if verbose:
        print(f"[After ] Missing values in '{target_col}': {remaining_nans}")

    return df

In [ ]:
impd_train_df = impute_values(df = train_df, target_col = "speed", verbose = True)
impd_val_df = impute_values(df = val_df, target_col = "speed", verbose = True)
impd_test_df = impute_values(df = test_df, target_col = "speed",    verbose = True)

print("\n\nComplete dataset:")
impd_df = impute_values(df = df, target_col = "speed", verbose = True) # but I will not use this, because it is imputed without any split, so there will be data leakage, but I am just doing it to check the overall imputation of the complete dataset

In [ ]:
# saving the prepared datasets with missing imputed 


# saving in the csv format
impd_train_df.to_csv(datasets / "solar_wind_speed_train_df.csv", index=False)
impd_val_df.to_csv(datasets / "solar_wind_speed_val_df.csv", index=False)
impd_test_df.to_csv(datasets / "solar_wind_speed_test_df.csv", index=False)
